# mean et argmax : réduire le bon axe

Choisir ce qui est regroupé et ce qui doit rester dans le résultat.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#selection-diffusion-reduction).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous isolons les réductions sur des identifiants simples, pas sur des mesures acoustiques. Trois lignes [1,2], [3,4], [5,6] rendent les deux axes faciles à distinguer.

| Demande | Calculs manuels | Valeurs et forme |
| --- | --- | --- |
| Une moyenne par colonne, dim=0 | (1+3+5)/3 ; (2+4+6)/3 | [3,4], (2,) |
| Une moyenne par ligne, dim=1 | (1+2)/2 ; (3+4)/2 ; (5+6)/2 | [1,5 ; 3,5 ; 5,5], (3,) |
| dim=0 avec keepdim=True | Les mêmes moyennes par colonne | [[3,4]], (1,2) |

dim identifie l’axe regroupé. keepdim préserve une place pour cet axe, pas ses anciennes valeurs.

## Démonstration 1 · Calculer les regroupements

Prévoir le nombre de valeurs avant d’appeler une méthode.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
# Indices de cases distincts : trois extraits, deux mesures.


### Calculer l’état suivant

X possède trois lignes et deux colonnes. Nous nommons ces axes extraits et mesures pour l’exercice.

In [4]:
X = torch.tensor([[1., 2.], [3., 4.], [5., 6.]])


In [5]:
observer('X', X)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]


### Calculer l’état suivant

mean(dim=0) additionne les trois positions d’une colonne puis divise par trois.

In [6]:
par_mesure = X.mean(dim=0)


In [7]:
observer('par_mesure', par_mesure)

par_mesure forme (2,) type torch.float32 requires_grad False
  valeurs : [3.0, 4.0]


### Calculer l’état suivant

mean(dim=1) additionne deux mesures par extrait puis divise par deux.

In [8]:
par_extrait = X.mean(dim=1)


In [9]:
observer('par_extrait', par_extrait)

par_extrait forme (3,) type torch.float32 requires_grad False
  valeurs : [1.5, 3.5, 5.5]


### Calculer l’état suivant

keepdim=True préserve une ligne de taille un. Les valeurs restent 3 et 4.

In [10]:
reference = X.mean(dim=0, keepdim=True)


In [11]:
observer('reference', reference)

reference forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[3.0, 4.0]]


### Lire les sorties

Nous retrouvons les scores du classifieur. argmax garde un indice au lieu d’une moyenne. dim=0 cherche cette fois l’extrait gagnant dans chaque classe.

In [12]:
print("dim0", par_mesure.tolist(), tuple(par_mesure.shape))
print("dim1", par_extrait.tolist(), tuple(par_extrait.shape))
print("keepdim", reference.tolist(), tuple(reference.shape))


dim0 [3.0, 4.0] (2,)
dim1 [1.5, 3.5, 5.5] (3,)
keepdim [[3.0, 4.0]] (1, 2)


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [13]:
scores = torch.tensor([[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]])


In [14]:
observer('scores', scores)

scores forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [15]:
print("classe par extrait", scores.argmax(dim=1).tolist())
print("extrait par classe", scores.argmax(dim=0).tolist())

classe par extrait [0, 0, 0]
extrait par classe [1, 0]


Le même numéro d’axe peut être réduit par plusieurs méthodes. Il faut aussi choisir si nous voulons une moyenne, une somme ou la position d’un maximum.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Choisissez dim pour obtenir une moyenne par ligne, soit [1,5 ; 3,5 ; 5,5].

In [16]:
exo_x = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
exo_dim = ...  # Remplacer les points par votre réponse.
if exo_dim is Ellipsis or exo_dim is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(exo_x.mean(dim=exo_dim),torch.tensor([1.5,3.5,5.5]))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Nous rassemblons les deux colonnes de chaque ligne ; les trois lignes restent distinctes.

</details>

### Exercice 2

Prévoyez argmax(dim=0) sur cette grille.

In [17]:
exo_x = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
exo_indices = ...  # Remplacer les points par votre réponse.
if exo_indices is Ellipsis or exo_indices is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_indices),exo_x.argmax(dim=0))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Chaque colonne compare ses trois lignes. Les deux maxima sont en ligne 2.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
exo_dim = 1
if exo_dim is Ellipsis or exo_dim is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    torch.testing.assert_close(exo_x.mean(dim=exo_dim),torch.tensor([1.5,3.5,5.5]))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
exo_indices = torch.tensor([2,2])
if exo_indices is Ellipsis or exo_indices is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.equal(torch.as_tensor(exo_indices),exo_x.argmax(dim=0))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [18]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()